In [1]:
import numpy as np
import pandas as pd
import warnings
import math
from datetime import timedelta
from glob import glob
import re
from pathlib import Path

In [2]:
class Meta:
    def __init__(self, VarNames, fileUnits, fileDepth):
        self.VarNames = VarNames
        self.fileUnits = fileUnits
        self.fileDepth = fileDepth

In [3]:
class Prev():
    def __init__(self):
        pass

    def update(self, meta, dateData, numericData, uniquedatetime_Length): 
        self.VarNames = meta.VarNames
        self.fileDepth = meta.fileDepth #corrected initialization (previously fileUnits)
        self.dateData = dateData
        self.numericData = numericData
        self.uniquedatetime_Length = uniquedatetime_Length #corrected variable spelling

In [4]:
# UNDERSTOOD
def readCSVFile(filepath):
    T = pd.read_csv(filepath)
    return T

In [5]:
# UNDERSTOOD
def unitConverter(fileUnits):
    if 'knots' in fileUnits:
        convertFactor = 0.51444448824222
    elif 'cm/sec' in fileUnits:
        convertFactor = 0.01
    else:
        warnings.warn('File defines water speed neither in knots nor cm/s. '
                      'Conversion factor of 1 is given by default. '
                      'Please Check file\'s units.')
        convertFactor = 1
    return convertFactor

In [6]:
# UNDERSTOOD
def extractData(T):
    dateData = pd.to_datetime(T.iloc[:, 0])
    numericaData = T.iloc[:, 1:]
    return dateData, numericaData

In [7]:
# UNDERSTOOD -- FIXED CODE FOR EFFICIENCY (regex implementation)
def extractMetaData(T, filename):
    VarNames = list(T.columns)

    speed_col  = next((name for name in VarNames if 'Speed' in name), None)
    if not speed_col:
        raise ValueError(f"{filename} does not contain 'Speed' column")

    # Extract units within the CSV file
    unit_match = re.search(r"Speed\s*[\(_]?([^\)]+)[\)]?", speed_col, re.IGNORECASE)
    fileUnits = unit_match.group(1).strip() if unit_match else ""

    # Extract depth from CSV file name
    depth_match = re.search(r"_(.*?)-", filename)
    fileDepth = depth_match.group(1) if depth_match else ""

    return Meta(VarNames, fileUnits, fileDepth)

In [8]:
# UNDERSTOOD
def valVarNames(current, previous, filePrev, fileCurr):
    if not current == previous:
        raise TypeError(f"Column mismatch between {filePrev}s and {fileCurr}s.")

In [9]:
# CHECK THIS CODE -- return variable mismatch
def initializeTimeSeries(dateData, numericData):

    #modified computation of dateDiff
    dateDiff = dateData.diff().dropna()
    if not dateDiff.eq(dateDiff.iloc[0]).all(): 
        warnings.warn("There is uneven date interval within current dataset, and so first interval is chosen by default.")

    dateInterval = dateDiff.iloc[0]

    dateOut = dateData
    dataOut = numericData
    dateOut_unique = dateData
    uniquedatetime_Length = len(dateData)

    return dateOut, dataOut, dateInterval, uniquedatetime_Length, dateOut_unique

In [10]:
# CHECK THIS CODE
def processTimeSeries(prev, dateData, numericData, fileDepth, fileCurrName, filePrevName, dateInterval):
    if (
        dateData[0] > prev.dateData.iloc[-1]
        and abs(dateInterval - (dateData[0] - prev.dateData.iloc[-1])) < timedelta(seconds = 1)
        and fileDepth == prev.fileDepth
    ):
        uniquedatetime = np.unique(pd.concat([prev.dateData, dateData], axis=0))
        uniquedatetime_Length = len(uniquedatetime)

        if not uniquedatetime_Length == prev.uniquedatetime_Length:
            timeDiff = abs(uniquedatetime_Length - prev.uniquedatetime_Length)
            timeDiff_datetime = timeDiff * dateInterval
            warnings.warn(f"There is a difference of {timeDiff} date entries between files {fileCurrName} and {filePrevName}."
                          f"The previous file ranges from {prev.dateData[0]} to {prev.dateData.iloc[-1]}."
                          f"The current file ranges from {dateData[0]} to {dateData.iloc[-1]}."
                          f"This number of entries correspond to a time difference of {timeDiff_datetime}.")
            user_concatenate = input("Do you approve the concatenation of these datasets?"
                                     "Please write '1' to approve or '0' to disapprove: ")

            if user_concatenate == "1":
                dateOut_unique = uniquedatetime[0:uniquedatetime_Length]
            else:
                dateOut_unique = uniquedatetime[0:prev.uniquedatetime_Length]

        dateOut = uniquedatetime
        dataOut = pd.concat([prev.numericData, numericData], axis=0, ignore_index=True)

    elif dateData.equals(prev.dateData): #corrected syntax spelling
        dateOut = prev.dateData
        dataOut = numericData
    elif not dateData.equals(prev.dateData) and fileDepth == prev.fileDepth:
        warnings.warn(f"Date mismatch between files {filePrevName} and {fileCurrName}.")
        dateOut = dateData
        dataOut = numericData
    else:
        dateOut = dateData
        dataOut = numericData

    try:
        uniquedatetime_Length
    except NameError:
        uniquedatetime_Length = prev.uniquedatetime_Length

    try:
        dateOut_unique
    except NameError:
        dateOut_unique = prev.dateOut_unique

    return dateOut, dateOut_unique, dataOut, uniquedatetime_Length

In [11]:
# UNDERSTOOD -- FIXED CODE FOR EFFICIENCY (regex implementation)
def extractSiteID(filenames):
    extraced_IDs = []

    for name in filenames:
        match = re.match(r'^([a-zA-Z]+)(\d+)', name)
        if match:
            extraced_IDs.append(match.group(1) + match.group(2))
        else:
            extraced_IDs.append("")
    if not extraced_IDs or any(s != extraced_IDs[0] for s in extraced_IDs):
        warnings.warn('There is a mismatch of site ID within provided data.')
        print('User-defined ID requested for plotting: ')
        return inputID()
    return extraced_IDs[0]

In [12]:
# UNDERSTOOD
def inputID():
    user_decision = input('Do you want to continue by defining the site ID? (Y/N): ')

    if user_decision == 'N':
        print('Exiting from FolderRead.'
              'recommendation to revise data in files.')
        return

    siteID = input('\n Please define the site ID to appear in plots (e.g. LIS1001): ')
    print('Continuing with user')
    return siteID

In [13]:
# UNDERSTOOD -- FIXED CODE FOR EFFICIENCY (regex implementation, simplified decimal creation)
def findDepth(filenames, files_num):
    fileDepths = []
    depthUnits = []

    pattern = re.compile(r'_(\d+)[a-zA-Z]+(\d+)([a-zA-Z]+)-?')

    for name in filenames: #corrected syntax spelling
        match = pattern.search(name)
        if not match:
            raise ValueError(
                'waterDepth:incorrectFormat',
                f"Error in file format for '{name}'. \n"
                'File format must list same units after site ID.\n'
                "Acceptable formats are the following: 'LIS1001_05m76cm' or 'LIS1001_18ft09df'."
            )

        integer_part, decimal_part, unit = match.groups()

        depth_value = float(f"{integer_part}.{decimal_part}")

        fileDepths.append(depth_value)
        depthUnits.append(unit)

    if len(set(depthUnits)) > 1:
        raise ValueError(
            'waterDepth:incorrectFormat',
            'Error in file format.\nFile format must list same units across all files.'
        )

    depth_units = depthUnits[0] if depthUnits else ""
    waterDepth = np.double(np.unique(fileDepths))

    return waterDepth, depth_units


In [14]:
def alignDataLengths(dataCells, targetLength):
    alignedCells = []

    for item in dataCells:
        row_count = len(item)

        if row_count >= targetLength:
            aligned_item = item.iloc[:targetLength]
            alignedCells.append(aligned_item)
    return alignedCells

In [15]:
def FolderReadCSV(folderpath):
    # preprocessing step 1
    files = glob(folderpath)
    files_num = len(files) 

    dataCells = [] 
    dateCells = []
    filenames = []

    prev = Prev() 

    for i in range(files_num):  
        filepath = files[i] 
        filenames.append(Path(files[i]).name) #generally extracts the name of each file from path  

        #read CSV file into a table 
        T = readCSVFile(filepath) 

        #extract metadata 
        meta = extractMetaData(T, filenames[i]) 

        if (i > 0) and (convertFactor in locals()):
            valVarNames(meta.VarNames, prev.VarNames, filenames[i-1], filenames[i]) 
        else:
            convertFactor = unitConverter(meta.fileUnits) 

        dateData, numericData = extractData(T) 

        if (i > 0) and (hasattr(prev, "dateData")): 
            dateOut, dateOut_unique, dataOut, uniquedatetime_Length = processTimeSeries(prev, dateData, numericData, meta.fileDepth, filenames[i], filenames[i-1], dateInterval)
            if not (np.array_equal(dateOut_unique, prev.dateOut_unique)): 
                warnings.warn("There is mismatch between unique dates.") 
                dateOut_uniqueSwitch = input("Write '1' for the new set of dates, or '0' to keep the old set of dates: ") 
                
                if dateOut_uniqueSwitch == '1': 
                    prev.dateOut_unique = dateOut_unique
                elif dateOut_uniqueSwitch == '0': 
                    dateOut_unique = prev.dateOut_unique 
                else: 
                    raise ValueError("Invalid input. Input must be numeric '1' or '0'.") #add this to first instance of input() as well
        else: 
            dateOut, dataOut, dateInterval, uniquedatetime_Length, dateOut_unique = initializeTimeSeries(dateData, numericData)
            prev.dateOut_unique = dateOut_unique 
            
        #assignment
        if (i % 2 == 1): 
            dateCells.append(dateOut) 
            dataCells.append(dataOut) 

        #update of prev object 
        prev.update(meta, dateData, numericData, uniquedatetime_Length) #removed assignment for clarity 

    #preprocessing step 2
    VarNames = prev.VarNames

    #extract site ID
    siteID = extractSiteID(filenames) 
    
    #gather water depths into list
    seadepths, depth_units = findDepth(filenames, files_num) 

    #daytime manipulation 
    DMY = dateOut_unique 
    dataCells = alignDataLengths(dataCells, len(DMY)) 
    return dataCells, dateCells, files_num, VarNames, DMY, dateInterval, convertFactor, siteID, seadepths, depth_units

In [16]:
dataCells, dateCells, files_num, VarNames, DMY, dateInterval, convertFactor, siteID, seadepths, depth_units = FolderReadCSV('C:\\Users\\ehsia\\Desktop\\SBU\\Tidal Data\\*.csv')

C:\Users\ehsia\AppData\Local\Temp\ipykernel_22284\3659608515.py:14: UserWarning: There is a difference of 4265 date entries between files LIS1016_05m76cm-2010-07-10.csv and LIS1016_05m76cm-2010-06-09.csv.The previous file ranges from 2010-06-09 20:12:00 to 2010-07-09 23:54:00.The current file ranges from 2010-07-10 00:00:00 to 2010-07-27 18:24:00.This number of entries correspond to a time difference of 17 days 18:30:00.
  warnings.warn(f"There is a difference of {timeDiff} date entries between files {fileCurrName} and {filePrevName}."


Do you approve the concatenation of these datasets?Please write '1' to approve or '0' to disapprove:  1


C:\Users\ehsia\AppData\Local\Temp\ipykernel_22284\2635004265.py:32: UserWarning: There is mismatch between unique dates.
  warnings.warn("There is mismatch between unique dates.")


Write '1' for the new set of dates, or '0' to keep the old set of dates:  1


C:\Users\ehsia\AppData\Local\Temp\ipykernel_22284\3659608515.py:14: UserWarning: There is a difference of 1 date entries between files LIS1016_25m76cm-2026-07-10.csv and LIS1016_25m76cm-2010-06-09.csv.The previous file ranges from 2010-06-09 20:12:00 to 2010-07-09 23:54:00.The current file ranges from 2010-07-10 00:00:00 to 2010-07-27 18:30:00.This number of entries correspond to a time difference of 0 days 00:06:00.
  warnings.warn(f"There is a difference of {timeDiff} date entries between files {fileCurrName} and {filePrevName}."


Do you approve the concatenation of these datasets?Please write '1' to approve or '0' to disapprove:  0


In [20]:
seadepths_length = len(seadepths)
seadepths_range = np.arange(0,seadepths_length,1, dtype=np.int64)
for i in seadepths_range:
    print(f'Depth {i:2d} of {seadepths_length-1:2d} = {seadepths[i]:2.2f} {depth_units}')
    if seadepths[i] > 0:
        seadepths[i] = -1*seadepths[i]
        
seatop = np.max(seadepths)
seabottom = np.min(seadepths)
print('seatop = {:2f} and seabottom = {:2f}'.format(seatop,seabottom))

Depth  0 of 16 = 5.76 cm
Depth  1 of 16 = 7.77 cm
Depth  2 of 16 = 9.78 cm
Depth  3 of 16 = 11.77 cm
Depth  4 of 16 = 13.78 cm
Depth  5 of 16 = 15.76 cm
Depth  6 of 16 = 17.77 cm
Depth  7 of 16 = 19.78 cm
Depth  8 of 16 = 21.76 cm
Depth  9 of 16 = 23.77 cm
Depth 10 of 16 = 25.76 cm
Depth 11 of 16 = 27.77 cm
Depth 12 of 16 = 29.78 cm
Depth 13 of 16 = 31.76 cm
Depth 14 of 16 = 33.77 cm
Depth 15 of 16 = 35.78 cm
Depth 16 of 16 = 37.77 cm
seatop = -5.760000 and seabottom = -37.770000


In [19]:
for name in VarNames:
    if name == 'Speed':
        #velStart = i # This syntax captures the variable name, but not the index
        magStart = name
    elif name == 'Dir':
        dirStart = name
        #dirStart = i
    #else: # This can contain an error to announce no detection
    #    print("This loop didn't detect velocity magnitude or direction.")

dataCells = np.array(dataCells)
velMag = dataCells[:,:,0::2] # This format enables access into nested arrays' alternating columns
#print(np.size(velMag,0))
#print(velMag)
velDir = dataCells[:,:,1::2]
#print(velDir)

# It may be that these lines are redundant; however, it may be better to remove NAN values BEFORE velMag and 
if np.any(np.isnan(velMag)):
    print("Removing nan values from velocity magnitude")
    velMag_noNAN = velMag[~np.isnan(velMag)]

In [45]:
velMag = velMag*convertFactor # Unit conversion
velDir_depthAvg = np.nanmean(velDir,0) # Depth averaging of velocity direction

#removed redundant initialization
velDir_cos = np.cos(np.deg2rad(velDir[:,:,0])).T 
velDir_sin = np.sin(np.deg2rad(velDir[:,:,0])).T 

eas_depthAvg = np.nanmean(velMag*velDir_cos,0)
nor_depthAvg = np.nanmean(velMag*velDir_sin,0)